# SDF 探索笔记本

`fogleman/sdf` 用法速览。每节都是一个独立的玩法，按顺序 Shift+Enter 即可。

第一次跑前请确认：
1. 已经 `conda activate sdf`
2. 在 `sdf-main/` 根目录跑过 `pip install -e .`
3. 选择 kernel = `sdf` (或 `Python 3`)

3D 渲染用 [k3d](https://k3d-jupyter.org)，鼠标拖动旋转、滚轮缩放、右键平移。


## 0. 公共 import 和 helper

所有后面的 cell 依赖这一格。

In [4]:
import numpy as np
import k3d
from sdf import *  # 引入所有 primitive 和 op


def show(f, **kw):
    """把 SDF 生成 mesh 并在 k3d 里显示。kw 透传给 generate()。"""
    pts = np.array(f.generate(verbose=False, **kw), dtype=np.float32)
    idx = np.arange(len(pts), dtype=np.uint32).reshape(-1, 3)
    plot = k3d.plot(grid_visible=True, camera_auto_fit=True)
    plot += k3d.mesh(pts, idx, color=0x88ccff, side='double',
                     flat_shading=False)
    plot.display()
    return plot


def show_many(items, cols=3):
    """同一个 plot 里横向排几个 SDF。items: [(name, sdf), ...]"""
    plot = k3d.plot(grid_visible=False, camera_auto_fit=True)
    for i, (name, f) in enumerate(items):
        pts = np.array(f.generate(verbose=False), dtype=np.float32)
        col = (i % cols)
        row = (i // cols)
        # 平移每个 mesh 让它们排开
        offset = np.array([col * 4.0, -row * 4.0, 0.0], dtype=np.float32)
        pts = pts + offset
        idx = np.arange(len(pts), dtype=np.uint32).reshape(-1, 3)
        plot += k3d.mesh(pts, idx, color=0x88ccff, side='double', name=name)
    plot.display()
    return plot


## 1. Hello SDF —— 球 ∩ 盒

README 里的开场白。三行写出经典的"球减三个方向圆柱"模型。

In [6]:
f = sphere(1) & box(1.5)
c = cylinder(0.5)
f -= c.orient(X) | c.orient(Y) | c.orient(Z)
show(f)


Output()

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

**也可以存成 STL 拿到 MeshLab / 3D 打印软件里：**

In [ ]:
f.save("out.stl", samples=2**20)  # 默认 2**22, 这里调小一些更快


## 2. 所有 3D 基本体一览

把 `d3.py` 里的 primitive 全部摆出来对比。

In [7]:
gallery = [
    ("sphere",          sphere(1)),
    ("box",             box(1.5)),
    ("rounded_box",     rounded_box((1.5, 1.5, 1.5), 0.3)),
    ("wireframe_box",   wireframe_box((1.5, 1.5, 1.5), 0.08)),
    ("torus",           torus(1, 0.3)),
    ("capsule",         capsule(-Z, Z, 0.5)),
    ("capped_cylinder", capped_cylinder(-Z, Z, 0.7)),
    ("rounded_cylinder",rounded_cylinder(0.7, 0.2, 1.5)),
    ("capped_cone",     capped_cone(-Z, Z, 1.0, 0.4)),
    ("rounded_cone",    rounded_cone(0.7, 0.3, 1.6)),
    ("ellipsoid",       ellipsoid((0.6, 0.9, 1.2))),
    ("pyramid",         pyramid(1.2)),
]
show_many(gallery, cols=4)


Output()

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

## 3. 五种正多面体 (Platonic solids)

In [ ]:
platonic = [
    ("tetrahedron",  tetrahedron(1)),
    ("octahedron",   octahedron(1)),
    ("dodecahedron", dodecahedron(1)),
    ("icosahedron",  icosahedron(1)),
]
show_many(platonic, cols=4)


## 4. 布尔运算 (CSG)

SDF 的杀手锏。`|` 并、`&` 交、`-` 差。

In [ ]:
a = box((3, 3, 0.5))
b = sphere(1.0)

show_many([
    ("a | b (union)",        a | b),
    ("a & b (intersection)", a & b),
    ("a - b (difference)",   a - b),
], cols=3)


## 5. Smooth 布尔 —— SDF 的魔法

普通 CSG 拼出来的接缝是棱角分明的。SDF 用 `k` 参数给布尔加一层"软过渡"，做出有机融合的效果——这是传统 mesh CSG 做不到的。

In [ ]:
a = box((3, 3, 0.5))
b = sphere(1.0)

show_many([
    ("hard union",        a | b),
    ("smooth k=0.1",      a | b.k(0.1)),
    ("smooth k=0.25",     a | b.k(0.25)),
    ("smooth k=0.5",      a | b.k(0.5)),
], cols=4)


**用 ipywidgets 拖 slider 实时看 k 的影响：**

In [ ]:
from ipywidgets import interact, FloatSlider

@interact(k=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.25))
def _(k=0.25):
    a = box((3, 3, 0.5))
    b = sphere(1.0)
    f = a | b.k(k) if k > 0 else a | b
    show(f)


## 6. 平移 / 旋转 / 缩放 / orient

`orient(axis)` 特别好用：把"原本指向 +Z 的形状"转到任意方向，省去手算旋转矩阵。

In [ ]:
c = capped_cylinder(-Z, Z, 0.25)
f = c.orient(X) | c.orient(Y) | c.orient(Z)  # 三轴十字
show(f)


In [ ]:
from sdf import pi

f = (capped_cylinder(-Z, Z, 0.5)
     .rotate(pi / 4, X)
     .translate((1, 0, 0)))
show(f)


## 7. 阵列 —— 无限 / 有限重复 / 圆周阵列

`repeat` 在指定方向无限或有限次重复一个 SDF。
`circular_array` 绕 Z 轴排成一圈。注意 `circular_array` 内部只算 2 次 SDF，比真复制 N 份快得多。

In [ ]:
# 8 根胶囊围成一圈
f = capped_cylinder(-Z*0.6, Z*0.6, 0.15).circular_array(8, 1.2)
show(f)


In [ ]:
# 在 X/Y 方向无限重复一个球
f = sphere(0.4).repeat((1.2, 1.2, 0)) & box((6, 6, 1))
show(f)


## 8. 形变：twist / bend / shell / wrap_around

SDF 的另一个杀手锏：可以在隐函数空间里直接做扭转、弯曲、抽壳，无需重新建模。

In [ ]:
show_many([
    ("twist",       box((1, 1, 2)).twist(pi / 2)),
    ("bend",        box((2, 0.5, 0.5)).bend(0.8)),
    ("shell",       sphere(1).shell(0.05) & plane(-Z)),  # 半球壳
    ("dilate 0.1",  box((1, 1, 1)).dilate(0.1)),
    ("erode 0.1",   box((1, 1, 1)).erode(0.1)),
    ("elongate",    sphere(0.5).elongate((0.5, 0, 0))),
], cols=3)


## 9. 2D 基本体 + extrude / revolve

`d2.py` 里的 2D SDF 可以 `.extrude(h)` 拉成柱，或 `.revolve(offset)` 绕 Y 轴旋成回转体。

In [ ]:
show_many([
    ("hexagon extrude",      hexagon(1).extrude(0.5)),
    ("rounded_rect extrude", rounded_rectangle((1.5, 1), 0.3).extrude(0.4)),
    ("polygon extrude",      polygon([(0,1),(1,0),(0.5,-1),(-0.5,-1),(-1,0)]).extrude(0.3)),
    ("hexagon revolve",      hexagon(0.5).revolve(2)),
], cols=2)


**`extrude_to`：起点到终点是不同 2D 形状的过渡（rect → circle）**

In [ ]:
f = rectangle(2).extrude_to(circle(1), 2, ease.in_out_quad)
show(f)


## 10. 文字 → 3D

把字符串当 SDF 用，可以直接拉伸或刻在盒子上。

In [ ]:
FONT = '/System/Library/Fonts/Helvetica.ttc'  # macOS 系统字体
TEXT = 'SDF!'

w, h = measure_text(FONT, TEXT)
plate = rounded_box((w + 1, h + 1, 0.25), 0.1)
letters = text(FONT, TEXT).extrude(1)
f = plate - letters  # 刻字
show(f)


## 11. show_slice —— 看 SDF 长啥样

调试自定义 SDF 时很有用。x=0 切片 → 看 yz 平面上每点到表面的距离。
红/蓝色 = 距离正负，零等值线就是表面。

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

f = sphere(1) | box(1.5).translate((1, 1, 0)).k(0.3)
f.show_slice(z=0)


## 12. 自己写 SDF

任何一个"输入 (N,3) 点 → 输出 (N,) 距离"的函数都能注册成 SDF。
下面这个例子是 IQ 网站上的"实心心形（绕 Z 轴旋转面）"的 SDF：

In [ ]:
@sdf3
def heart(scale=1.0):
    """一个心形，沿 Z 拉伸一点。"""
    def f(p):
        x = p[:, 0] / scale
        y = p[:, 1] / scale
        z = p[:, 2] / scale
        # 经典心形隐式: (x^2 + 9/4 y^2 + z^2 - 1)^3 - x^2 z^3 - 9/80 y^2 z^3
        # 这不是真正的 SDF, 用 sign(F) * |F|^(1/3) 近似下
        F = (x*x + 9/4*y*y + z*z - 1)**3 - x*x*z*z*z - 9/80*y*y*z*z*z
        return np.sign(F) * np.abs(F) ** (1/3) * scale
    return f

show(heart(1.0))


## 13. 综合作品 —— 一个有机花瓶

把上面学的全用上：六边形 → revolve → twist → smooth shell。

In [ ]:
base = hexagon(1.2).revolve(0.1)              # 六棱柱回转体, 像花瓶
base = base.twist(0.7)                          # 拧一下
base = base & slab(z0=-1.5, z1=1.5)             # 高度切到 -1.5 ~ 1.5
base = base.shell(0.08)                         # 抽 0.08 厚的壁
base = base & slab(z1=1.4)                      # 顶部开口
show(base, samples=2**20)


---

## 下一步

玩到这里你应该对 SDF 能做什么有感觉了。后面的 JS 移植就照着这些玩法对齐 API 即可。

**一些可以在这个 notebook 里继续探索的方向：**
- 用 `repeat` + 多层 smooth union 做 [blobby.py](../examples/blobby.py) 那种有机生物
- 把 `image()` 接进来做带浮雕的牌子 (examples/image.py)
- 写自己的扭曲算子，在 `@op3` 装饰器下注册成方法
- 把 `k3d.mesh` 换成 `trimesh.Trimesh` 看顶点法线、做 mesh 后处理
